In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
REPO_URL = 'https://github.com/Dweeb1578/voicemos-2026.git'
REPO_DIR = '/content/voicemos-2026'
CKPT_DIR = '/content/drive/MyDrive/voicemos2026/checkpoints'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
os.chdir(REPO_DIR)
os.makedirs(CKPT_DIR, exist_ok=True)

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-r', 'requirements.txt', '-q'], check=True)


In [ ]:
!python data/download.py --output data/datasets --datasets bvcc tmhint audiomos25t3

In [ ]:
!python -m data.build_manifests --data_dir data/datasets --output_dir data/manifests

In [ ]:
# Pre-extract frozen Whisper encoder outputs (one-time, ~1-2h on T4).
# Saves ~47 GB to data/encoder_cache/. Skip if cache already exists.
import os
if not os.path.exists('data/encoder_cache') or len(os.listdir('data/encoder_cache')) < 100:
    !python -m data.cache_features \
        --manifests data/manifests/pretrain_train.csv data/manifests/pretrain_dev.csv \
        --cache_dir data/encoder_cache \
        --whisper_model openai/whisper-medium \
        --batch_size 32
else:
    print(f"Cache exists ({len(os.listdir('data/encoder_cache'))} files), skipping extraction.")

In [ ]:
!python -m src.train --config configs/pretrain.yaml

## Experiment: Layer-12 features (intermediate Whisper layer)
Research shows intermediate SSL layers correlate better with perceptual quality than the final layer. Extract layer 12 of 24 from Whisper-medium and train a parallel run to compare SRCC.

In [ ]:
import os
if not os.path.exists('data/encoder_cache_layer12') or len(os.listdir('data/encoder_cache_layer12')) < 100:
    !python -m data.cache_features \
        --manifests data/manifests/pretrain_train.csv data/manifests/pretrain_dev.csv \
        --cache_dir data/encoder_cache_layer12 \
        --whisper_model openai/whisper-medium \
        --layer 12 \
        --batch_size 8
else:
    print(f"Layer-12 cache exists ({len(os.listdir('data/encoder_cache_layer12'))} files), skipping.")

In [ ]:
!python -m src.train --config configs/pretrain_layer12.yaml

In [ ]:
import shutil, os
CKPT_DIR = '/content/drive/MyDrive/voicemos2026/checkpoints'
shutil.copy('checkpoints/pretrain/best.pt', f'{CKPT_DIR}/pretrain_best.pt')
print('Checkpoint saved to Drive.')


In [ ]:
import torch
from torch.utils.data import DataLoader
from src.model import WhisperMOSNet
from src.dataset import MOSDataset
from src.evaluate import compute_metrics

device = torch.device('cuda')

def eval_checkpoint(ckpt_path, cache_dir, label):
    model = WhisperMOSNet(whisper_model='openai/whisper-medium', proj_dim=256, dropout=0.1).to(device)
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    model.train(False)

    ds = MOSDataset('data/manifests/pretrain_dev.csv',
                    whisper_model='openai/whisper-medium', cache_dir=cache_dir)
    loader = DataLoader(ds, batch_size=16, shuffle=False, num_workers=2)

    acr_preds, acr_targets = [], []
    with torch.no_grad():
        for batch in loader:
            inp = batch.get('input_features', None)
            enc = batch.get('encoder_feats', None)
            if inp is not None: inp = inp.to(device)
            if enc is not None: enc = enc.to(device)
            acr, _ = model(inp, batch['waveform'].to(device), encoder_feats=enc)
            acr_preds.extend(acr.cpu().tolist())
            acr_targets.extend(batch['acr'].tolist())

    m = compute_metrics(acr_preds, acr_targets)
    beat = 'BEAT' if m['srcc'] > 0.780 else 'not yet'
    print(f"[{label}]  SRCC={m['srcc']:.4f}  LCC={m['lcc']:.4f}  MSE={m['mse']:.4f}  ({beat} baseline)")
    return m

# Final layer
if os.path.exists('checkpoints/pretrain/best.pt'):
    eval_checkpoint('checkpoints/pretrain/best.pt', 'data/encoder_cache', 'Final layer (24)')

# Layer 12
if os.path.exists('checkpoints/pretrain_layer12/best.pt'):
    eval_checkpoint('checkpoints/pretrain_layer12/best.pt', 'data/encoder_cache_layer12', 'Layer 12')

## Dev-set submission (VoiceMOS 2026 Track 1)

Generate a CodaBench `predictions.csv` for the official dev set
(`urgent-challenge/vmc2026-track1-dev`: 1008 ACR + 2520 CCR samples).

ACR is predicted directly by the pretrained ACR head; CCR is derived as
`clamp(acr_b - acr_a, -3, 3)` from the same head (the CCR head isn't trained
yet). This validates the full submission round-trip and yields a baseline
UTT-SRCC. Upload the resulting `submission.zip` to CodaBench.

In [ ]:
# Faster HF downloads: authenticated + hf_transfer parallel downloader.
# Store your HF *read* token in Colab Secrets (key icon, left sidebar) as
# HF_TOKEN, with notebook access enabled. Never hardcode the token here --
# this notebook is committed to a public repo.
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab Secrets.")
except Exception as e:
    print("No HF_TOKEN secret found -- downloads will be unauthenticated/slow.", e)

!pip install -q hf_transfer
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"  # carried into the !python subprocess below

In [ ]:
# 1. Download + materialize the dev set (FLAC) and build manifests.
#    -> data/manifests/dev_acr.csv (1008), data/manifests/dev_ccr.csv (2520)
!python data/prepare_dev.py --output data/datasets/track1_dev

In [ ]:
# 2. Run inference and build the submission.
#    Uses the pretrained checkpoint synced to Drive in the cell above.
CKPT = '/content/drive/MyDrive/voicemos2026/checkpoints/pretrain_best.pt'
!python -m scripts.predict_dev \
    --checkpoint "{CKPT}" \
    --config configs/pretrain.yaml \
    --acr-manifest data/manifests/dev_acr.csv \
    --ccr-manifest data/manifests/dev_ccr.csv \
    --output predictions.csv \
    --zip submission.zip

# Sanity-check + download
import pandas as pd
df = pd.read_csv('predictions.csv')
print(df['sample_id'].str.contains('-acr_').sum(), 'ACR +',
      df['sample_id'].str.contains('-ccr_').sum(), 'CCR =', len(df), 'rows')
print('ACR range:', df[df.sample_id.str.contains('-acr_')].pred_score.agg(['min','max']).tolist())
print('CCR range:', df[df.sample_id.str.contains('-ccr_')].pred_score.agg(['min','max']).tolist())
from google.colab import files
files.download('submission.zip')